# ETF Risk Scoring Model

This notebook combines market-risk, holdings, sector, country and currency measures into transparent sub-scores and an overall ETF risk score from 0 to 100.

In [1]:
import numpy as np
import pandas as pd

risk_metrics = pd.read_csv(
    "../data/processed/etf_risk_metrics.csv"
)

holdings_metrics = pd.read_csv(
    "../data/processed/etf_holdings_metrics_base.csv"
)

sector_concentration = pd.read_csv(
    "../data/processed/etf_sector_concentration.csv"
)

country_concentration = pd.read_csv(
    "../data/processed/etf_country_concentration.csv"
)

print("Risk metrics:", risk_metrics.shape)
print("Holdings metrics:", holdings_metrics.shape)
print("Sector concentration:", sector_concentration.shape)
print("Country concentration:", country_concentration.shape)

Risk metrics: (20, 15)
Holdings metrics: (20, 9)
Sector concentration: (12, 6)
Country concentration: (19, 6)


In [2]:
holdings_features = holdings_metrics[
    [
        "Ticker",
        "Number_of_Holdings",
        "Holdings_Count_Risk",
        "Main_Currency_Exposure",
        "Currency_Risk_Level",
        "Currency_Risk_Score"
    ]
].copy()

sector_features = sector_concentration.rename(
    columns={
        "Largest_Exposure": "Largest_Sector_Exposure",
        "Number_of_Categories": "Number_of_Sectors",
        "HHI": "Sector_HHI",
        "Effective_Categories": "Effective_Sectors"
    }
)[
    [
        "Ticker",
        "Largest_Sector_Exposure",
        "Number_of_Sectors",
        "Sector_HHI",
        "Effective_Sectors"
    ]
]

country_features = country_concentration.rename(
    columns={
        "Largest_Exposure": "Largest_Country_Exposure",
        "Number_of_Categories": "Number_of_Countries",
        "HHI": "Country_HHI",
        "Effective_Categories": "Effective_Countries"
    }
)[
    [
        "Ticker",
        "Largest_Country_Exposure",
        "Number_of_Countries",
        "Country_HHI",
        "Effective_Countries"
    ]
]

scoring_data = (
    risk_metrics
    .merge(holdings_features, on="Ticker", how="left")
    .merge(sector_features, on="Ticker", how="left")
    .merge(country_features, on="Ticker", how="left")
)

print("Scoring dataset shape:", scoring_data.shape)

print("\nMissing values in scoring features:")
print(
    scoring_data[
        [
            "Annualised_Volatility",
            "Maximum_Drawdown",
            "Daily_CVaR_95",
            "Beta",
            "Holdings_Count_Risk",
            "Currency_Risk_Score",
            "Sector_HHI",
            "Country_HHI"
        ]
    ].isna().sum()
)

scoring_data.head()

Scoring dataset shape: (20, 28)

Missing values in scoring features:
Annualised_Volatility    0
Maximum_Drawdown         0
Daily_CVaR_95            0
Beta                     0
Holdings_Count_Risk      0
Currency_Risk_Score      0
Sector_HHI               8
Country_HHI              1
dtype: int64


,ETF_ID,ETF_Name,Ticker,Provider,Risk_Category,Asset_Class,Annualised_Volatility,Maximum_Drawdown,Daily_VaR_95,Daily_CVaR_95,...,Currency_Risk_Level,Currency_Risk_Score,Largest_Sector_Exposure,Number_of_Sectors,Sector_HHI,Effective_Sectors,Largest_Country_Exposure,Number_of_Countries,Country_HHI,Effective_Countries
0,VWRP,Vanguard FTSE All-World UCITS ETF,VWRP.L,Vanguard,Medium,Equity,0.128639,-0.176420,-0.012678,-0.018968,...,Medium,50,35.10,11.0,0.1853,5.40,61.70,16.0,0.3937,2.54
1,SWDA,iShares Core MSCI World UCITS ETF,SWDA.L,iShares,Medium,Equity,0.228266,-0.274175,-0.013244,-0.022911,...,Medium,50,30.27,11.0,0.1589,6.29,72.45,6.0,0.5470,1.83
2,VUAG,Vanguard S&P 500 UCITS ETF,VUAG.L,Vanguard,Medium,Equity,0.142655,-0.208796,-0.014310,-0.021214,...,Medium,50,37.45,11.0,0.1931,5.18,100.00,1.0,1.0000,1.00
3,ISF,iShares Core FTSE 100 UCITS ETF,ISF.L,iShares,Medium,Equity,0.125398,-0.131797,-0.012368,-0.019460,...,Low,25,28.61,12.0,0.1559,6.42,100.00,1.0,1.0000,1.00
4,EQQQ,Invesco EQQQ Nasdaq-100 UCITS ETF,EQQQ.L,Invesco,Higher,Equity,0.192915,-0.281650,-0.019855,-0.027479,...,Medium,50,68.51,9.0,0.5005,2.00,100.00,1.0,1.0000,1.00


In [3]:
def min_max_score(series):
    """
    Converts a numeric series to a 0–100 scale.
    Higher original values produce higher risk scores.
    Missing values remain missing.
    """
    minimum = series.min()
    maximum = series.max()

    if maximum == minimum:
        return pd.Series(50, index=series.index)

    return 100 * (series - minimum) / (maximum - minimum)


# Convert negative loss measures into positive risk magnitudes
scoring_data["Drawdown_Risk_Value"] = (
    scoring_data["Maximum_Drawdown"].abs()
)

scoring_data["CVaR_Risk_Value"] = (
    scoring_data["Daily_CVaR_95"].abs()
)

# Negative beta is treated as zero systematic equity-market exposure
scoring_data["Beta_Risk_Value"] = (
    scoring_data["Beta"].clip(lower=0)
)

# Market-risk scores
scoring_data["Volatility_Score"] = min_max_score(
    scoring_data["Annualised_Volatility"]
)

scoring_data["Drawdown_Score"] = min_max_score(
    scoring_data["Drawdown_Risk_Value"]
)

scoring_data["CVaR_Score"] = min_max_score(
    scoring_data["CVaR_Risk_Value"]
)

scoring_data["Beta_Score"] = min_max_score(
    scoring_data["Beta_Risk_Value"]
)

# Exposure-concentration scores
scoring_data["Sector_Concentration_Score"] = min_max_score(
    scoring_data["Sector_HHI"]
)

scoring_data["Country_Concentration_Score"] = min_max_score(
    scoring_data["Country_HHI"]
)

score_columns = [
    "Ticker",
    "Volatility_Score",
    "Drawdown_Score",
    "CVaR_Score",
    "Beta_Score",
    "Holdings_Count_Risk",
    "Currency_Risk_Score",
    "Sector_Concentration_Score",
    "Country_Concentration_Score"
]

scoring_data[score_columns].round(2)

,Ticker,Volatility_Score,Drawdown_Score,CVaR_Score,Beta_Score,Holdings_Count_Risk,Currency_Risk_Score,Sector_Concentration_Score,Country_Concentration_Score
0,VWRP.L,37.47,21.96,45.62,67.98,18.07,50,5.04,30.33
1,SWDA.L,71.89,40.62,56.68,65.81,29.80,50,1.96,47.95
2,VUAG.L,42.31,28.14,51.92,72.20,39.94,50,5.94,100.00
3,ISF.L,36.35,13.45,47.01,45.14,57.41,25,1.61,100.00
4,EQQQ.L,59.68,42.05,69.50,92.11,57.41,50,41.78,100.00
5,VEUR.L,40.21,19.55,48.31,57.10,39.70,50,0.00,0.00
6,VJPN.L,47.32,23.30,55.06,55.88,40.55,50,2.65,100.00
7,VFEG.L,45.77,24.69,54.43,57.17,23.44,75,7.02,9.72
8,ICHN.AS,93.50,86.13,99.86,63.90,38.49,75,1.32,100.00
9,IUIT.L,75.12,52.16,84.29,100.00,60.64,50,100.00,100.00


In [4]:
market_weights = {
    "Volatility_Score": 0.30,
    "Drawdown_Score": 0.30,
    "CVaR_Score": 0.25,
    "Beta_Score": 0.15
}

concentration_weights = {
    "Holdings_Count_Risk": 0.40,
    "Sector_Concentration_Score": 0.30,
    "Country_Concentration_Score": 0.30
}

overall_weights = {
    "Market_Risk_Score": 0.60,
    "Concentration_Risk_Score": 0.30,
    "Currency_Risk_Score": 0.10
}


def weighted_average_available(row, weights):
    """
    Calculates a weighted average using only available measures.
    Weights are automatically rebalanced when a metric is not applicable.
    """
    available = {
        column: weight
        for column, weight in weights.items()
        if pd.notna(row[column])
    }

    if not available:
        return np.nan

    total_weight = sum(available.values())

    return sum(
        row[column] * weight
        for column, weight in available.items()
    ) / total_weight


scoring_data["Market_Risk_Score"] = scoring_data.apply(
    weighted_average_available,
    axis=1,
    weights=market_weights
)

scoring_data["Concentration_Risk_Score"] = scoring_data.apply(
    weighted_average_available,
    axis=1,
    weights=concentration_weights
)

scoring_data["Overall_Risk_Score"] = scoring_data.apply(
    weighted_average_available,
    axis=1,
    weights=overall_weights
)

scoring_data[
    [
        "ETF_Name",
        "Ticker",
        "Asset_Class",
        "Market_Risk_Score",
        "Concentration_Risk_Score",
        "Currency_Risk_Score",
        "Overall_Risk_Score"
    ]
].sort_values(
    "Overall_Risk_Score",
    ascending=False
).round(2)

,ETF_Name,Ticker,Asset_Class,Market_Risk_Score,Concentration_Risk_Score,Currency_Risk_Score,Overall_Risk_Score
11,iShares Global Clean Energy Transition UCITS ETF,INRG.L,Equity,95.61,59.39,75,82.68
9,iShares S&P 500 Information Technology Sector ...,IUIT.L,Equity,74.26,84.26,50,74.83
8,iShares MSCI China UCITS ETF,ICHN.AS,Equity,88.44,45.79,75,74.30
4,Invesco EQQQ Nasdaq-100 UCITS ETF,EQQQ.L,Equity,61.71,65.50,50,61.67
18,iShares Physical Gold ETC,SGLN.L,Gold,41.47,100.00,50,59.88
10,iShares S&P 500 Health Care Sector UCITS ETF,IUHC.L,Equity,38.50,85.16,50,53.64
19,iShares Developed Markets Property Yield UCITS...,IWDP.L,Property,44.90,68.59,50,52.52
12,Vanguard U.K. Gilt UCITS ETF,VGOV.L,Bond,36.84,78.10,25,48.04
1,iShares Core MSCI World UCITS ETF,SWDA.L,Equity,57.80,26.89,50,47.75
2,Vanguard S&P 500 UCITS ETF,VUAG.L,Equity,44.95,47.76,50,46.30


In [5]:
scoring_data["Risk_Rank"] = (
    scoring_data["Overall_Risk_Score"]
    .rank(method="min", ascending=False)
    .astype(int)
)

scoring_data["Risk_Band"] = pd.cut(
    scoring_data["Overall_Risk_Score"],
    bins=[-0.01, 20, 40, 60, 80, 100],
    labels=[
        "Very Low",
        "Low",
        "Moderate",
        "High",
        "Very High"
    ]
)

risk_ranking = scoring_data[
    [
        "Risk_Rank",
        "ETF_ID",
        "ETF_Name",
        "Ticker",
        "Asset_Class",
        "Market_Risk_Score",
        "Concentration_Risk_Score",
        "Currency_Risk_Score",
        "Overall_Risk_Score",
        "Risk_Band"
    ]
].sort_values("Risk_Rank")

risk_ranking[
    [
        "Market_Risk_Score",
        "Concentration_Risk_Score",
        "Currency_Risk_Score",
        "Overall_Risk_Score"
    ]
] = risk_ranking[
    [
        "Market_Risk_Score",
        "Concentration_Risk_Score",
        "Currency_Risk_Score",
        "Overall_Risk_Score"
    ]
].round(2)

risk_ranking

,Risk_Rank,ETF_ID,ETF_Name,Ticker,Asset_Class,Market_Risk_Score,Concentration_Risk_Score,Currency_Risk_Score,Overall_Risk_Score,Risk_Band
11,1,INRG,iShares Global Clean Energy Transition UCITS ETF,INRG.L,Equity,95.61,59.39,75,82.68,Very High
9,2,IUIT,iShares S&P 500 Information Technology Sector ...,IUIT.L,Equity,74.26,84.26,50,74.83,High
8,3,ICHN,iShares MSCI China UCITS ETF,ICHN.AS,Equity,88.44,45.79,75,74.30,High
4,4,EQQQ,Invesco EQQQ Nasdaq-100 UCITS ETF,EQQQ.L,Equity,61.71,65.50,50,61.67,High
18,5,SGLN,iShares Physical Gold ETC,SGLN.L,Gold,41.47,100.00,50,59.88,Moderate
10,6,IUHC,iShares S&P 500 Health Care Sector UCITS ETF,IUHC.L,Equity,38.50,85.16,50,53.64,Moderate
19,7,IWDP,iShares Developed Markets Property Yield UCITS...,IWDP.L,Property,44.90,68.59,50,52.52,Moderate
12,8,VGOV,Vanguard U.K. Gilt UCITS ETF,VGOV.L,Bond,36.84,78.10,25,48.04,Moderate
1,9,SWDA,iShares Core MSCI World UCITS ETF,SWDA.L,Equity,57.80,26.89,50,47.75,Moderate
2,10,VUAG,Vanguard S&P 500 UCITS ETF,VUAG.L,Equity,44.95,47.76,50,46.30,Moderate


In [6]:
driver_columns = {
    "Market_Risk_Score": "Market risk",
    "Concentration_Risk_Score": "Concentration risk",
    "Currency_Risk_Score": "Currency risk"
}

scoring_data["Main_Risk_Driver"] = (
    scoring_data[list(driver_columns)]
    .idxmax(axis=1)
    .map(driver_columns)
)

risk_ranking = scoring_data[
    [
        "Risk_Rank",
        "ETF_ID",
        "ETF_Name",
        "Ticker",
        "Asset_Class",
        "Market_Risk_Score",
        "Concentration_Risk_Score",
        "Currency_Risk_Score",
        "Overall_Risk_Score",
        "Risk_Band",
        "Main_Risk_Driver"
    ]
].sort_values("Risk_Rank")

risk_ranking

,Risk_Rank,ETF_ID,ETF_Name,Ticker,Asset_Class,Market_Risk_Score,Concentration_Risk_Score,Currency_Risk_Score,Overall_Risk_Score,Risk_Band,Main_Risk_Driver
11,1,INRG,iShares Global Clean Energy Transition UCITS ETF,INRG.L,Equity,95.605031,59.391603,75,82.680499,Very High,Market risk
9,2,IUIT,iShares S&P 500 Information Technology Sector ...,IUIT.L,Equity,74.256707,84.256000,50,74.830824,High,Concentration risk
8,3,ICHN,iShares MSCI China UCITS ETF,ICHN.AS,Equity,88.438853,45.791151,75,74.300657,High,Market risk
4,4,EQQQ,Invesco EQQQ Nasdaq-100 UCITS ETF,EQQQ.L,Equity,61.709718,65.496929,50,61.674910,High,Concentration risk
18,5,SGLN,iShares Physical Gold ETC,SGLN.L,Gold,41.471987,100.000000,50,59.883192,Moderate,Concentration risk
10,6,IUHC,iShares S&P 500 Health Care Sector UCITS ETF,IUHC.L,Equity,38.495362,85.156000,50,53.644017,Moderate,Concentration risk
19,7,IWDP,iShares Developed Markets Property Yield UCITS...,IWDP.L,Property,44.903378,68.588571,50,52.518598,Moderate,Concentration risk
12,8,VGOV,Vanguard U.K. Gilt UCITS ETF,VGOV.L,Bond,36.839951,78.104535,25,48.035331,Moderate,Concentration risk
1,9,SWDA,iShares Core MSCI World UCITS ETF,SWDA.L,Equity,57.796031,26.892176,50,47.745271,Moderate,Market risk
2,10,VUAG,Vanguard S&P 500 UCITS ETF,VUAG.L,Equity,44.947858,47.759425,50,46.296542,Moderate,Currency risk


In [7]:
# Save the user-facing ETF ranking
risk_ranking.to_csv(
    "../data/processed/etf_risk_ranking.csv",
    index=False
)

# Save the complete modelling dataset for validation
scoring_data.to_csv(
    "../data/processed/etf_scoring_components.csv",
    index=False
)

print("Risk-ranking file saved:", risk_ranking.shape)
print("Full scoring file saved:", scoring_data.shape)
print(
    "Missing overall scores:",
    scoring_data["Overall_Risk_Score"].isna().sum()
)

Risk-ranking file saved: (20, 11)
Full scoring file saved: (20, 43)
Missing overall scores: 0
